Importing cellular automata & optimization classes, and other stuff

In [1]:
import os
import sys
import shutil

from typing import List, Type, Callable, Dict
from numpy import int32
from numpy._typing import NDArray
import importlib

sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))

#from algorithm.blender import Lattice, clear_initial
from algorithm.genetic import Optimizer, Mutator, ArbitraryRulesetMutator
from algorithm.objectives import sampleobj, surface_to_vol, surfacecalc

import numpy as np
import pandas as pd

import time

Setting up optimizer and data logging code

In [ ]:
def log_mutation(data_list: List[Dict], mutations: List[tuple[List,List]], objective_val: float):
    """
    Given the data list reference, the mutation set, and the objective value after applying it, add it to the data logging list
    """
    ic_cell_pos = []
    ic_state_old = []
    ic_state_new = []
    srt_cell_pos = []
    srt_state_old = []
    srt_state_new = []

    """
    Important difference from original genetic algorithm: 
    ic_mut is a 4 membered list showing the mutated position in the IC.
    Example: [0,0,0,1], meaning that at IC pos (0,0) the state 0 is modified to be 1.
    srt_mut is a 6 membered list showing the mutated position in the ruleset.
    Example: [0,0,0,0,0,1], meaning that at SRT rule pos (0,0,0) index 0 the rule 0 is modified to be 1.
    For a 2 state SRT there are 18 different cells for mutation: 0 - 17.
    """
    for ic_mut in mutations[0]:
        ic_cell_pos.append(tuple(ic_mut[0:2]))
        ic_state_old.append(ic_mut[2])
        ic_state_new.append(ic_mut[3])
    for srt_mut in mutations[1]:
        srt_cell_pos.append(tuple(srt_mut[0:4]))
        srt_state_old.append(srt_mut[4])
        srt_state_new.append(srt_mut[5])
    # print(f"Number of IC mutations: {len(mutations[0])}, Number of SRT mutations: {len(mutations[1])}", end='\r')
    data_list.append({
        "ic_cell_pos": np.array(ic_cell_pos), 
        "ic_state_old": np.array(ic_state_old), 
        "ic_state_new": np.array(ic_state_new), 
        "srt_cell_pos": np.array(srt_cell_pos), 
        "srt_state_old": np.array(srt_state_old), 
        "srt_state_new": np.array(srt_state_new), 
        "objective": objective_val,
    })

itlogs = [0]

def run_experiment(iters: int, grid_sz: int, opt_func: Callable[[NDArray[int32]], int], ic_num_mutate: int, srt_num_mutate: int, rule_mutate_prob: float, strict: bool = False, num_strict: bool = True, states: int = 2, ic_enable: bool = True, srt_enable: bool = True):
    """
    Runs an experiment with the below hyperparameters:

    :param iters: The number of iterations the mutation algorithm (updating both IC and SRT) is going to run for
    :param grid_sz: The size of the square grid that we're going to update each iteration
    :param opt_func: The functions that gives the performance metric we're going to optimize
    :param srt_num_mutate: The number of SRT cells for which we're going to mutate the rule applied, each iteration
    :param ic_num_mutate: The number of IC cells for which we're going to mutate the rule applied, each iteration
    :param rule_mutate_prob: The probability, for each neighbor state tensor of the rule of a cell that's selected to be mutated, the final state is mutated
    :param strict: Whether SRT mutations are chosen by cell then rule or by rule directly; for more info, see mutations.py
    :param num_strict: Whether the number of SRT/IC cells mutated will be constant per iteration or variable; for more info, see mutations.py
    :param states: The number of states the NSTICA will run on; for more info, see nstica.py
    :param ic_enable: Whether the IC will be mutated
    :param srt_enable: Whether the SRT will be mutated

    Params ruleset_mutator_class and rule_set have been deleted due to previous deletion of the RulesetMutator class.
    """

    # RESOLVED: separate SRT and IC mutations to have a certain number of each
    # RESOLVED: add a flag to enable doing only SRT or only IC mutations in an iteration (in optimizer step, and then propagate into mutator)

    ruleset_mutator = ArbitraryRulesetMutator(grid_size=grid_sz, mutate_p=1/(grid_sz**2) * (srt_num_mutate+ic_num_mutate), rule_mutate_p=rule_mutate_prob, strict=strict, num_strict = num_strict, ic_ct = ic_num_mutate, srt_ct = srt_num_mutate, states=states, ic_enable=ic_enable, srt_enable = srt_enable)

    optim = Optimizer(mutator=ruleset_mutator, objective=lambda grid: opt_func(grid))
    """
    Pandas Dataframe used to log experiment data is:

    ic_cell_pos (np.array) | ic_state_old (np.array) | ic_state_new (np.array) | srt_cell_pos (np.array) | srt_state_old (np.array) | srt_state_new (np.array) | objective (float)
    
    etc.

    initial state for IC is in entry 0 in ic_state_old, and SRT is in entry 0 in srt_state_old

    ic and srt mutation cell positions and states can have an extra dimension in the beginning to indicate they are batch updates
    """

    init_state = optim.state
    
    data_list = [{"ic_cell_pos": grid_sz, 
                  "ic_state_old": init_state[0], 
                  "ic_state_new": None, 
                  "srt_cell_pos": -1, 
                  "srt_state_old": init_state[1], 
                  "srt_state_new": None, 
                  "objective": 0}]

    for it in range(iters):
        # print(f"Iteration {it}", end='\r')
        # print(f"On iteration {it+1}...")
        accepted, new, old, mutations = optim.step()
        # data logging
        log_mutation(data_list, mutations, optim.objvalue)
        #itlogs.append(float(optim.objvalue))
        #if accepted:
        #   print(f"Got a better state: {optim.objvalue} at iteration {it}, which is {(100*optim.objvalue/(-98304)):.3f}% of the ground truth.")

    # print(data_list)
    df = pd.DataFrame(data_list)
    # print(df)
    return df

timelogs = []

def repeat_experiment(experiment_name: str, num_expers: int, *args):
    """
    Perform (sequentially) multiple experiments that return a Pandas DataFrame and save all the data

    :param experiment_name: The name of the experiment to save the file
    :param num_expers: Number of times to run the experiment (and save all the data in one file)
    :param *args: The arguments to be passed to the experiment function

    Within *args should be a states parameter, denoting the number of NSTICA states.
    """
    for i in range(num_expers):
        init = time.time()
        print(f'REPETITION {i}')
        ret_data = run_experiment(*args)
        timelogs.append(time.time() - init)
        print(f"Finished rep {i} in {time.time() - init}s")
        ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Setting up experiments and gathering data

In [6]:
#ITERATIONS_SET = [50, 100, 200, 500]
#GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 50
EXPERIMENT_NAME = "statetimetest"
#STATE_SET = [2, 3, 4, 5]
#NUMS_SET = [(1, 18), (2, 36), (5, 90), (10, 180), (20, 360), (50, 900), (100, 1800), (200, 3600), (500, 9000), (1000, 18000), (2000, 36000)]
COND_SET = [(50, 10, 2), (50, 20, 2), (50, 32, 2), (50, 64, 2), (50, 100, 2), (100, 10, 2), (100, 20, 2), (100, 32, 2), (100, 64, 2), (100, 100, 2), (200, 10, 2), (200, 20, 2), (200, 32, 2), (200, 64, 2), (200, 100, 2), (500, 10, 2), (500, 20, 2), (500, 32, 2), (500, 64, 2), (500, 100, 2), (50, 10, 3), (50, 20, 3), (50, 32, 3), (50, 64, 3), (100, 10, 3), (100, 20, 3), (100, 32, 3), (100, 64, 3), (200, 10, 3), (200, 20, 3), (200, 32, 3), (200, 64, 3), (500, 10, 3), (500, 20, 3), (500, 32, 3), (50, 10, 4), (50, 20, 4), (50, 32, 4), (100, 10, 4), (100, 20, 4), (100, 32, 4), (200, 10, 4), (200, 20, 4), (200, 32, 4), (500, 10, 4), (500, 20, 4), (500, 32, 4), (50, 10, 5), (50, 20, 5), (50, 32, 5), (100, 10, 5), (100, 20, 5), (100, 32, 5), (200, 10, 5), (200, 20, 5), (500, 10, 5), (500, 20, 5), (50, 10, 6), (50, 20, 6), (100, 10, 6), (100, 20, 6), (200, 10, 6), (200, 20, 6), (500, 10, 6)]
for conditions in COND_SET:
#for iters in ITERATIONS_SET:
    #for grid_sz in GRID_SIZE_SET:
        #for states in STATE_SET:
            #for nums in NUMS_SET:
                iters = conditions[0]
                grid_sz = conditions[1]
                states = conditions[2]
                print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID AND {states} STATES")
                #Default probability for Strict Mode = 2/3; default probability for Non-Strict Mode = 5/384
                #Default number of IC mutations = 20, default number of SRT mutations = 240
                repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, surfacecalc, grid_sz, grid_sz**2, 2/3, True, True, states, True, True)
                print(itlogs)

RUNNING EXPERIMENT statetimetest WITH 50 ITERATIONS AND 10 SIZE GRID AND 2 STATES
REPETITION 0
Finished rep 0 in 0.4392688274383545s
REPETITION 1


/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 1 in 0.4338645935058594s
REPETITION 2


/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 2 in 0.43149232864379883s
REPETITION 3


...

/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 1
Finished rep 1 in 215.91403937339783s


/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 2


KeyboardInterrupt: 

In [7]:
print(timelogs)

[0.4392673969268799, 0.43386292457580566, 0.431490421295166, 0.43245434761047363, 0.43239402770996094, 0.4300980567932129, 0.4333646297454834, 0.4322190284729004, 0.42182350158691406, 0.41488099098205566, 0.4118459224700928, 0.4121825695037842, 0.4171586036682129, 0.4190695285797119, 0.4154810905456543, 0.4213380813598633, 0.42362046241760254, 0.42319726943969727, 0.421431303024292, 0.4237806797027588, 0.41950297355651855, 0.4197554588317871, 0.4333040714263916, 0.4163944721221924, 0.4114565849304199, 0.42351675033569336, 0.4216740131378174, 0.42339181900024414, 0.4226536750793457, 0.416837215423584, 0.4207022190093994, 0.428328275680542, 0.42507123947143555, 0.4257171154022217, 0.42238378524780273, 0.4216268062591553, 0.42714834213256836, 0.425520658493042, 0.4282047748565674, 0.4190669059753418, 0.4205362796783447, 0.42412781715393066, 0.4222412109375, 0.42691969871520996, 0.42229127883911133, 0.42700934410095215, 0.42727184295654297, 0.426959753036499, 0.4269065856933594, 0.42669391

In [ ]:
print(itlogs)


In [8]:
#ITERATIONS_SET = [50, 100, 200, 500]
#GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 50
EXPERIMENT_NAME = "statetimetest"
#STATE_SET = [2, 3, 4, 5]
#NUMS_SET = [(1, 18), (2, 36), (5, 90), (10, 180), (20, 360), (50, 900), (100, 1800), (200, 3600), (500, 9000), (1000, 18000), (2000, 36000)]
COND_SET = [(100, 10, 4), (100, 20, 4), (100, 32, 4), (200, 10, 4), (200, 20, 4), (200, 32, 4), (500, 10, 4), (500, 20, 4), (500, 32, 4), (50, 10, 5), (50, 20, 5), (50, 32, 5), (100, 10, 5), (100, 20, 5), (100, 32, 5), (200, 10, 5), (200, 20, 5), (500, 10, 5), (500, 20, 5), (50, 10, 6), (50, 20, 6), (100, 10, 6), (100, 20, 6), (200, 10, 6), (200, 20, 6), (500, 10, 6)]
for conditions in COND_SET:
#for iters in ITERATIONS_SET:
    #for grid_sz in GRID_SIZE_SET:
        #for states in STATE_SET:
            #for nums in NUMS_SET:
                iters = conditions[0]
                grid_sz = conditions[1]
                states = conditions[2]
                print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID AND {states} STATES")
                #Default probability for Strict Mode = 2/3; default probability for Non-Strict Mode = 5/384
                #Default number of IC mutations = 20, default number of SRT mutations = 240
                repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, surfacecalc, grid_sz, grid_sz**2, 2/3, True, True, states, True, True)
                print(itlogs)

RUNNING EXPERIMENT statetimetest WITH 100 ITERATIONS AND 10 SIZE GRID AND 4 STATES
REPETITION 0
Finished rep 0 in 1.645003318786621s
REPETITION 1


/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 1 in 1.6489675045013428s
REPETITION 2


/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 2 in 1.6444220542907715s
REPETITION 3


...

/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 7
Finished rep 7 in 268.55560278892517s


/tmp/ipykernel_5920/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 8


KeyboardInterrupt: 

In [ ]:
#CPU MutVar Trial Timelogs: Be careful for times of 50IT-64GS-4S trials 0-1 and 200IT-10GS-6S trials 0-7 at end during analysis!!
print(timelogs)

[0.4392673969268799, 0.43386292457580566, 0.431490421295166, 0.43245434761047363, 0.43239402770996094, 0.4300980567932129, 0.4333646297454834, 0.4322190284729004, 0.42182350158691406, 0.41488099098205566, 0.4118459224700928, 0.4121825695037842, 0.4171586036682129, 0.4190695285797119, 0.4154810905456543, 0.4213380813598633, 0.42362046241760254, 0.42319726943969727, 0.421431303024292, 0.4237806797027588, 0.41950297355651855, 0.4197554588317871, 0.4333040714263916, 0.4163944721221924, 0.4114565849304199, 0.42351675033569336, 0.4216740131378174, 0.42339181900024414, 0.4226536750793457, 0.416837215423584, 0.4207022190093994, 0.428328275680542, 0.42507123947143555, 0.4257171154022217, 0.42238378524780273, 0.4216268062591553, 0.42714834213256836, 0.425520658493042, 0.4282047748565674, 0.4190669059753418, 0.4205362796783447, 0.42412781715393066, 0.4222412109375, 0.42691969871520996, 0.42229127883911133, 0.42700934410095215, 0.42727184295654297, 0.426959753036499, 0.4269065856933594, 0.42669391

In [3]:
ITERATIONS_SET = [50, 100, 200, 500]
GRID_SIZE_SET = [10, 20, 32, 64]
NUM_REPEAT = 50
EXPERIMENT_NAME = "comparative" #Directly comparable to naive implementation
STATE_SET = [2]
#NUMS_SET = [(1, 18), (2, 36), (5, 90), (10, 180), (20, 360), (50, 900), (100, 1800), (200, 3600), (500, 9000), (1000, 18000), (2000, 36000)]
for iters in ITERATIONS_SET:
    for grid_sz in GRID_SIZE_SET:
        for states in STATE_SET:
            #for nums in NUMS_SET:
                print(f"RUNNING EXPERIMENT {EXPERIMENT_NAME} WITH {iters} ITERATIONS AND {grid_sz} SIZE GRID AND {states} STATES")
                #Default probability for Strict Mode = 2/3; default probability for Non-Strict Mode = 5/384
                #Default number of IC mutations = 20, default number of SRT mutations = 240
                repeat_experiment(f"{EXPERIMENT_NAME}_{iters}ITERS_{grid_sz}GRID", NUM_REPEAT, iters, grid_sz, surface_to_vol, 10, 10, 2/3, True, False, states, True, True)
                print(itlogs)

RUNNING EXPERIMENT comparative WITH 50 ITERATIONS AND 10 SIZE GRID AND 2 STATES
REPETITION 0
Finished rep 0 in 2.2940173149108887s


/tmp/ipykernel_1500235/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 1
Finished rep 1 in 0.6139016151428223s
REPETITION 2


/tmp/ipykernel_1500235/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


Finished rep 2 in 0.5337784290313721s
REPETITION 3


...

REPETITION 48
Finished rep 48 in 104.25316643714905s


/tmp/ipykernel_1500235/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


REPETITION 49
Finished rep 49 in 103.63836193084717s


/tmp/ipykernel_1500235/3296386025.py:121: PerformanceWarning: 
your performance may suffer as PyTables will pickle object types that it cannot
map directly to c-types [inferred_type->mixed-integer,key->block0_values] [items->Index(['ic_cell_pos', 'ic_state_old', 'ic_state_new', 'srt_cell_pos',
       'srt_state_old', 'srt_state_new', 'objective'],
      dtype='object')]

  ret_data.to_hdf(f'test/{states}/{experiment_name}_{i}_{states}states.h5', key='data', mode='a')


[0]


In [ ]:
#CPU Naive-Comparable Trial Timelogs
print(timelogs)

[2.2940151691436768, 0.6138994693756104, 0.5337765216827393, 0.6345517635345459, 0.6392796039581299, 0.6491756439208984, 0.6501944065093994, 0.5582680702209473, 0.6513357162475586, 0.6563417911529541, 0.6550185680389404, 0.5625050067901611, 0.6627521514892578, 0.6677758693695068, 0.6675477027893066, 0.6630098819732666, 0.5756347179412842, 0.66188645362854, 0.6680517196655273, 0.6647493839263916, 0.688326358795166, 0.577322244644165, 0.6577036380767822, 0.6546025276184082, 0.6512441635131836, 0.5675332546234131, 0.6580266952514648, 0.6630091667175293, 0.6507787704467773, 0.6467878818511963, 0.5657038688659668, 0.6480984687805176, 0.6468441486358643, 0.6552038192749023, 0.5591726303100586, 0.6474399566650391, 0.6575617790222168, 0.651620626449585, 0.6533229351043701, 0.5634782314300537, 0.6469123363494873, 0.6524999141693115, 0.6556341648101807, 0.6616189479827881, 0.56142258644104, 0.672292947769165, 0.6606001853942871, 0.6744353771209717, 0.6618053913116455, 0.5659976005554199, 2.20864